# Detailed Multi-Recall Pipeline Notebook

这个 notebook 将召回全流程拆成多个步骤，方便逐段调试和观察中间结果。

In [ ]:
import os
import pickle

from metrics import metrics_recall
from recall_pipeline import (
    RECALL_METHODS,
    cold_start_items,
    combine_recall_results,
    get_eval_frames,
    load_artifacts,
    run_embedding_recall,
    run_itemcf_recall,
    run_youtube_usercf_recall,
    youtubednn_u2i_dict,
)
from utils.data_utils import get_user_hist_item_info_dict


## 1. 配置

In [ ]:
data_path = './tcdata/'
save_path = './temp_results/'

offline = False
metric_recall = True
use_sample = True
sample_nums = 10000

enable_methods = list(RECALL_METHODS)

weight_dict = {
    'itemcf_sim_itemcf_recall': 1.0,
    'embedding_sim_item_recall': 1.0,
    'youtubednn_recall': 1.0,
    'youtubednn_usercf_recall': 1.0,
    'cold_start_recall': 1.0,
}

print('Config loaded')
print(f'use_sample={use_sample}, sample_nums={sample_nums}, offline={offline}, metric_recall={metric_recall}')
print(f'enable_methods={enable_methods}')


## 2. 加载基础数据与物料

In [ ]:
artifacts = load_artifacts(
    data_path=data_path,
    save_path=save_path,
    offline=offline,
    use_sample=use_sample,
    sample_nums=sample_nums,
)

all_click_df = artifacts['all_click_df']
item_info_df = artifacts['item_info_df']
item_type_dict = artifacts['item_type_dict']
item_words_dict = artifacts['item_words_dict']
item_created_time_dict = artifacts['item_created_time_dict']
item_emb_df = artifacts['item_emb_df']

print(all_click_df.shape)
print(item_info_df.shape)
all_click_df.head()


## 3. 生成评估集 / 召回输入集

In [ ]:
recall_click_df, trn_last_click_df = get_eval_frames(all_click_df, metric_recall)
print(recall_click_df.shape)
if trn_last_click_df is not None:
    print(trn_last_click_df.shape)
recall_click_df.head()


## 4. embedding i2i 召回

In [ ]:
emb_i2i_sim, embedding_recall_dict = run_embedding_recall(
    recall_click_df,
    item_created_time_dict,
    item_emb_df,
    save_path,
)

print(len(embedding_recall_dict))
first_user = next(iter(embedding_recall_dict))
print(first_user, embedding_recall_dict[first_user][:10])


## 5. ItemCF 召回

In [ ]:
itemcf_recall_dict = run_itemcf_recall(
    recall_click_df,
    item_created_time_dict,
    emb_i2i_sim,
    save_path,
)

print(len(itemcf_recall_dict))
first_user = next(iter(itemcf_recall_dict))
print(first_user, itemcf_recall_dict[first_user][:10])


## 6. YoutubeDNN u2i 召回

In [ ]:
youtube_recall_dict, youtube_user_emb_dict = youtubednn_u2i_dict(
    recall_click_df,
    item_info_df,
    save_path,
)

print(len(youtube_recall_dict))
first_user = next(iter(youtube_recall_dict))
print(first_user, youtube_recall_dict[first_user][:10])


## 7. YoutubeDNN usercf 召回

In [ ]:
youtube_usercf_recall_dict = run_youtube_usercf_recall(
    recall_click_df,
    youtube_user_emb_dict,
    item_created_time_dict,
    emb_i2i_sim,
    save_path,
)

print(len(youtube_usercf_recall_dict))
first_user = next(iter(youtube_usercf_recall_dict))
print(first_user, youtube_usercf_recall_dict[first_user][:10])


## 8. 冷启动召回过滤

In [ ]:
all_click_with_info = all_click_df.merge(item_info_df, how='left', on='click_article_id')
user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict = get_user_hist_item_info_dict(all_click_with_info)
click_article_ids_set = set(all_click_df['click_article_id'].values)

cold_start_recall_dict = cold_start_items(
    embedding_recall_dict,
    user_hist_item_typs_dict,
    user_hist_item_ids_dict,
    user_hist_item_words_dict,
    user_last_item_created_time_dict,
    item_type_dict,
    item_words_dict,
    item_created_time_dict,
    click_article_ids_set,
    recall_item_num=100,
)

print(len(cold_start_recall_dict))
first_user = next(iter(cold_start_recall_dict))
print(first_user, cold_start_recall_dict[first_user][:10])


## 9. 汇总多路召回结果

In [ ]:
user_multi_recall_dict = {
    'itemcf_sim_itemcf_recall': itemcf_recall_dict,
    'embedding_sim_item_recall': embedding_recall_dict,
    'youtubednn_recall': youtube_recall_dict,
    'youtubednn_usercf_recall': youtube_usercf_recall_dict,
    'cold_start_recall': cold_start_recall_dict,
}

for method, recall_result in user_multi_recall_dict.items():
    print(method, len(recall_result))


## 10. 分路评估

In [ ]:
if metric_recall and trn_last_click_df is not None:
    for method, recall_result in user_multi_recall_dict.items():
        if recall_result:
            print(f'\n[{method}]')
            metrics_recall(
                recall_result,
                trn_last_click_df,
                topk=min(50, max(len(items) for items in recall_result.values()))
            )
else:
    print('metric_recall is disabled')


## 11. 多路融合

In [ ]:
final_recall_items_dict = combine_recall_results(
    user_multi_recall_dict,
    weight_dict=weight_dict,
    topk=150,
    save_path=save_path,
)

print(len(final_recall_items_dict))
first_user = next(iter(final_recall_items_dict))
print(first_user, final_recall_items_dict[first_user][:10])


## 12. 读取落盘结果检查

In [ ]:
final_path = os.path.join(save_path, 'final_recall_items_dict.pkl')
with open(final_path, 'rb') as f:
    final_recall_items_dict_from_disk = pickle.load(f)

print(final_path)
print(len(final_recall_items_dict_from_disk))
